# EDA 1

In [1]:
import scipy.io

# Path to the sample battery .mat file
file_path = '../Battery_DataSet/B0005.mat'

# Load the MATLAB file
b0005_data = scipy.io.loadmat(file_path)

# Let's see the top-level keys in the file
print("Top-level keys in B0005.mat:")
for key in b0005_data.keys():
    if not key.startswith('__'): # Ignore matlab internal variables
        top_var = b0005_data[key]
        print(f"- '{key}': {type(top_var)} of shape {getattr(top_var, 'shape', 'N/A')}")


Top-level keys in B0005.mat:
- 'B0005': <class 'numpy.ndarray'> of shape (1, 1)


In [2]:
# The actual data is inside the 'B0005' key. 
# It's a structured numpy array.
battery = b0005_data['B0005']

# Print its dtype (which shows the fields inside the MATLAB struct)
print("Fields inside 'B0005':")
print(battery.dtype.names)

# The 'cycle' field holds the operations.
cycles = battery[0, 0]['cycle']
print(f"\nNumber of cycles recorded: {cycles.size}")
print(f"Shape of the cycle array: {cycles.shape}")

# Let's look at the first cycle's fields
first_cycle = cycles[0, 0]
print("\nFields inside the first cycle:")
print(first_cycle.dtype.names)


Fields inside 'B0005':
('cycle',)

Number of cycles recorded: 616
Shape of the cycle array: (1, 616)

Fields inside the first cycle:
('type', 'ambient_temperature', 'time', 'data')


In [3]:
import numpy as np

# Let's inspect the first 5 operations to see the sequence
print("First 5 operations recorded:")
for i in range(5):
    cycle = cycles[0, i]
    # MATLAB strings load as single-element numpy arrays
    op_type = cycle['type'][0]
    temp = cycle['ambient_temperature'][0, 0]
    # The data field contains the measurements
    data = cycle['data']
    print(f"Cycle {i+1}: Type = {op_type}, Ambient Temp = {temp}C")
    print(f"  Available data fields: {data.dtype.names}")

# Let's see how many Discharge operations have 'Capacity' measured
capacity_records = 0
discharges = 0
for i in range(cycles.size):
    cycle = cycles[0, i]
    if cycle['type'][0] == 'discharge':
        discharges += 1
        data = cycle['data']
        if 'Capacity' in data.dtype.names:
            capacity_records += 1

print(f"\nOut of {discharges} total discharge cycles, {capacity_records} have recorded Capacity.")

First 5 operations recorded:
Cycle 1: Type = charge, Ambient Temp = 24C
  Available data fields: ('Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time')
Cycle 2: Type = discharge, Ambient Temp = 24C
  Available data fields: ('Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time', 'Capacity')
Cycle 3: Type = charge, Ambient Temp = 24C
  Available data fields: ('Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time')
Cycle 4: Type = discharge, Ambient Temp = 24C
  Available data fields: ('Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time', 'Capacity')
Cycle 5: Type = charge, Ambient Temp = 24C
  Available data fields: ('Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time')

Out of 168 total discharge cycles, 168 have recorded Capacity.


### Stage 2: Extracting the Discharge Capacity
**The Purpose:** The primary goal of analyzing this dataset is to predict the **State of Health (SoH)** and the **Remaining Useful Life (RUL)** of a battery. 

The `Capacity` (measured in Amp-hours) is the definitive measure of a battery's health. 
* As a battery charges and discharges repeatedly, its internal chemistry degrades, and it can hold less charge.
* NASA considered these batteries "dead" when their capacity faded by 30% (from 2.0 Ahr down to roughly 1.4 Ahr).

By extracting the overall `Capacity` for each of the 168 discharge cycles, we can plot a single line graph showing the battery "dying" over time. This flattened trend of `(Cycle Index, Capacity)` is exactly what a Machine Learning model needs to learn how rapidly a battery degrades!

In [5]:
import pandas as pd

# List to hold our flattened data
capacity_data = []

# We'll keep a counter specifically for discharge cycles (1 to 168)
discharge_cycle_number = 1

for i in range(cycles.size):
    cycle = cycles[0, i]
    op_type = cycle['type'][0]
    
    # We only care about discharge cycles because that's when total capacity is evaluated
    if op_type == 'discharge':
        
        # Access the inner data structure
        data = cycle['data']
        
        # In MATLAB structs loaded by scipy.io, the values are often deeply nested arrays.
        # data['Capacity'][0, 0] points to the actual scalar value.
        if 'Capacity' in data.dtype.names:
            capacity_val = data['Capacity'][0, 0][0, 0]
            
            capacity_data.append({
                'Discharge_Cycle': discharge_cycle_number,
                'Capacity_Ahr': capacity_val
            })
            
            discharge_cycle_number += 1

# Convert to a Pandas DataFrame
df_capacity = pd.DataFrame(capacity_data)

# Show the first 5 rows (Fresh Battery) and last 5 rows (Aged Battery)
print("Fresh Battery (First 5 Discharge Cycles):")
display(df_capacity.head())

print("\nAged Battery (Last 5 Discharge Cycles):")
display(df_capacity.tail())


Fresh Battery (First 5 Discharge Cycles):


,Discharge_Cycle,Capacity_Ahr
0,1,1.856487
1,2,1.846327
2,3,1.835349
3,4,1.835263
4,5,1.834646



Aged Battery (Last 5 Discharge Cycles):


,Discharge_Cycle,Capacity_Ahr
163,164,1.293464
164,165,1.288003
165,166,1.287453
166,167,1.309015
167,168,1.325079
